# 04 — Producción: limpieza de la producción mensual por pozo (2024–2026)

**Objetivo:** aplicar las decisiones de limpieza tomadas sobre los hallazgos del perfilado y generar una versión limpia,
**sin modificar los originales**.

| Dato | Valor |
| --- | --- |
| Archivos de origen | `data/raw/produccin-de-pozos-de-gas-y-petrleo-2024.csv`, `-2025.csv` y `-2026.csv` (no se modifican) |
| Diagnóstico | `03_produccion_perfilado.ipynb`: hallazgos **H-28 a H-44** y decisiones **D-13 a D-21** |
| Referencia | `data/clean/capitulo-iv-pozos_limpio.csv` (maestro de pozos limpio, generado por el notebook 02) |
| Política aplicada | `POLITICA_CALIDAD_DATOS.md`: pasos 3 a 6 (elegir la acción, aplicar y medir, validar, documentar) |
| Resultado | `data/clean/produccion_2024-2026_limpio.parquet` |
| Etapa siguiente | `05_eda.ipynb` — análisis exploratorio y gráficos |

**Criterio general** (aprobado el 13/10/2026): **ante la duda, marcar antes que eliminar**.
No se elimina ninguna fila ni se modifica ningún valor de producción: solo se renombran dos columnas
y se agregan columnas nuevas (una fecha y cuatro marcas).

**Trazabilidad:** las reglas siguen la numeración del notebook 02 (R-01 a R-20). Las de producción empiezan en **R-21**.
Cada regla indica qué decisión (D-xx) aplica y qué hallazgos (H-xx) resuelve.

## 1. Preparar el entorno

- **pandas** trabaja con tablas.
- **pyarrow** es la librería que usa pandas para leer y escribir archivos **Parquet** (decisión D-21).
- **hashlib** calcula la huella digital SHA-256 de un archivo, y **os** consulta su tamaño.

Mostramos las versiones porque forman parte de la reproducibilidad.

In [ ]:
import hashlib
import os

import pandas as pd
import pyarrow

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
print("Versión de pandas: ", pd.__version__)
print("Versión de pyarrow:", pyarrow.__version__)

CARPETA_RAW = "../data/raw"
ARCHIVOS = {
    2024: "produccin-de-pozos-de-gas-y-petrleo-2024.csv",
    2025: "produccin-de-pozos-de-gas-y-petrleo-2025.csv",
    2026: "produccin-de-pozos-de-gas-y-petrleo-2026.csv",
}
RUTA_MAESTRO = "../data/clean/capitulo-iv-pozos_limpio.csv"
RUTA_CLEAN = "../data/clean/produccion_2024-2026_limpio.parquet"

## 2. Origen: huella digital y carga

Igual que en el notebook 03:
- Calculamos la **huella SHA-256** de cada raw **antes** de leerlo. Al final la volvemos a calcular para probar que no cambió.
- Leemos **solo las 22 columnas** necesarias (`usecols`), para no agotar la memoria.
- Agregamos **`archivo_origen`**: cada fila queda vinculada a su archivo raw (linaje).

Guardamos también la **suma de producción de los raw** (`total_raw`): al final comprobamos que el archivo limpio suma exactamente lo mismo.
Es la prueba de que la limpieza no movió ni un m³.

In [ ]:
def huella_sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


COLUMNAS = [
    "idpozo", "sigla", "anio", "mes", "idempresa", "empresa",
    "prod_pet", "prod_gas", "tef",
    "tipo_de_recurso", "sub_tipo_recurso", "formacion", "formprod", "cuenca", "provincia",
    "areayacimiento", "tipoestado", "tipopozo",
    "rectificado", "habilitado", "fechaingreso", "fecha_data",
]

huellas_raw = {}
partes = []
for anio, nombre in ARCHIVOS.items():
    ruta = os.path.join(CARPETA_RAW, nombre)
    huellas_raw[nombre] = huella_sha256(ruta)
    parte = pd.read_csv(ruta, usecols=COLUMNAS, low_memory=False)
    parte["archivo_origen"] = nombre
    partes.append(parte)
    print(f"{nombre}: {len(parte):,} filas  |  SHA-256: {huellas_raw[nombre]}")

df = pd.concat(partes, ignore_index=True)
del partes

TOTAL_FILAS = len(df)
total_raw = df[["prod_pet", "prod_gas"]].sum()
print(f"\nTotal: {TOTAL_FILAS:,} filas  |  petróleo: {total_raw['prod_pet']:,.0f} m³  |  gas: {total_raw['prod_gas']:,.0f} miles de m³")

## 3. Registro de reglas

Cada regla se anota con: su código, la decisión que aplica, los hallazgos que resuelve,
la acción de la política (*Corregir*, *Marcar* o *Sin acción*), qué hace y **cuántas filas afectó**.
Al final, esta lista se convierte en la tabla de linaje. Es la misma función que en el notebook 02.

In [ ]:
registro = []


def registrar(regla, decision, hallazgos, accion, que_hace, filas):
    registro.append({
        "regla": regla, "decision": decision, "hallazgos": hallazgos,
        "accion": accion, "que_hace": que_hace, "filas_afectadas": int(filas),
    })
    print(f"{regla} ({decision}): {int(filas):,} filas afectadas ({filas / TOTAL_FILAS * 100:.2f}%)")

## 4. Aplicar las reglas

### R-21 · La unidad en el nombre de la columna (D-20)

La política dice: *nunca cambiar unidades en silencio; el nombre de la columna dice la unidad*.
Acá **no se convierte nada**: los valores siguen en las unidades de la fuente.
Solo se renombran las columnas para que la unidad quede a la vista y nadie sume petróleo con gas por error:

- `prod_pet` → `prod_pet_m3` (metros cúbicos)
- `prod_gas` → `prod_gas_miles_m3` (miles de metros cúbicos)

`rename` cambia solo el nombre; el contenido queda igual.

In [ ]:
df = df.rename(columns={"prod_pet": "prod_pet_m3", "prod_gas": "prod_gas_miles_m3"})
registrar("R-21", "D-20", "-", "Corregir", "Renombra prod_pet y prod_gas con su unidad (sin convertir valores)", TOTAL_FILAS)

### R-22 · Columna `fecha_mes` (D-22, aprobada el 14/10/2026)

El mes viene en dos columnas de números (`anio` y `mes`). Para graficar en el tiempo conviene una **fecha real**:
`fecha_mes` es el **primer día de cada mes** (por ejemplo, marzo de 2025 → 2025-03-01).

¿Por qué el primer día y no `fecha_data` (el último)? Porque es la convención habitual para series mensuales, y
porque `fecha_data` viene como texto. Las columnas originales `anio`, `mes` y `fecha_data` se conservan.

`pd.to_datetime` arma la fecha a partir de una tabla con columnas `year`, `month` y `day`.

In [ ]:
df["fecha_mes"] = pd.to_datetime(pd.DataFrame({"year": df["anio"], "month": df["mes"], "day": 1}))

assert df["fecha_mes"].notna().all(), "Hay meses que no se pudieron convertir a fecha"
registrar("R-22", "D-22", "-", "Corregir", "Agrega fecha_mes (primer día del mes) a partir de anio y mes", TOTAL_FILAS)
print("Rango:", df["fecha_mes"].min().date(), "a", df["fecha_mes"].max().date())

### R-23 · Marca de `tef` fuera de rango (D-13, H-38)

`tef` (tiempo efectivo de producción, en días) no puede ser negativo ni mayor que los días del mes.
En el perfilado: 2.074 filas se pasan por 1 día, 476 por más de un mes y 6 son negativas.

**Se marca y se conserva:** no hay un valor defendible para reemplazarlo, y ningún gráfico del EDA usa `tef`.
Si en el futuro un análisis lo usa (por ejemplo, producción por día efectivo), tendrá que excluir las filas marcadas.

`days_in_month` da los días de cada mes y contempla los años bisiestos.

In [ ]:
df["tef_fuera_de_rango"] = (df["tef"] < 0) | (df["tef"] > df["fecha_mes"].dt.days_in_month)
registrar("R-23", "D-13", "H-38", "Marcar", "Marca tef negativo o mayor que los días del mes; el valor se conserva",
          df["tef_fuera_de_rango"].sum())

### R-24 · Marca de producción negativa (D-14, H-36, H-37)

Nadie produce un volumen negativo, pero estas filas parecen **ajustes contables** de la empresa
(por ejemplo, Bentia en septiembre de 2025) o ceros con error de redondeo. Su peso en los totales es nulo.

**Se marca y se conserva:** cambiarlas a 0 sería modificar un dato de la fuente sin evidencia de cuál es el valor correcto.
En el histograma con escala logarítmica (EDA, gráfico 3) quedan afuera solas, porque el logaritmo de un negativo no existe.

In [ ]:
df["produccion_negativa"] = (df["prod_pet_m3"] < 0) | (df["prod_gas_miles_m3"] < 0)
registrar("R-24", "D-14", "H-36, H-37", "Marcar", "Marca petróleo o gas negativos; el valor se conserva",
          df["produccion_negativa"].sum())

### R-25 · Marca de pozo fuera del maestro (D-15, H-42)

Algunos `idpozo` de producción no existen en el maestro de pozos limpio. La mayoría son **pozos nuevos**
(dados de alta después de la versión del maestro); 29 producen desde 2024 y quedan sin explicar.

**Se marca y se conserva:** su producción es real; eliminarlos achicaría los totales.
La marca avisa que para esas filas no hay ficha del pozo (profundidad, coordenadas, etc.).

In [ ]:
maestro = pd.read_csv(RUTA_MAESTRO, usecols=["idpozo", "tipo_recurso"], low_memory=False)

df["fuera_del_maestro"] = ~df["idpozo"].isin(maestro["idpozo"])
registrar("R-25", "D-15", "H-42", "Marcar", "Marca idpozo que no está en el maestro de pozos limpio",
          df["fuera_del_maestro"].sum())

### R-26 · Marca de tipo de recurso inconsistente (D-18, H-39, H-43)

Dos situaciones sospechosas, que se marcan con una sola columna:

1. **H-39:** el pozo tiene **más de un** `tipo_de_recurso` en el período (sin contar "No informado").
2. **H-43:** el `tipo_de_recurso` de producción es **distinto** del `tipo_recurso` del maestro (solo cuando los dos tienen dato).

**Se marca, y el EDA usa el tipo del archivo de producción**: no tiene faltantes, mientras que el maestro tiene un 25% de nulos.

`transform("nunique")` cuenta los tipos de cada pozo y devuelve el resultado **en cada fila** del pozo,
así la marca queda en todas sus filas. `map` busca el tipo del maestro para cada `idpozo`.

In [ ]:
tipo_informado = df["tipo_de_recurso"].where(df["tipo_de_recurso"] != "No informado")
varios_tipos = tipo_informado.groupby(df["idpozo"]).transform("nunique") > 1

tipo_maestro = df["idpozo"].map(maestro.set_index("idpozo")["tipo_recurso"])
distinto_del_maestro = tipo_informado.notna() & tipo_maestro.notna() & (tipo_informado != tipo_maestro)

df["recurso_inconsistente"] = varios_tipos | distinto_del_maestro
print(f"H-39 (varios tipos en el período): {varios_tipos.sum():,} filas  |  H-43 (distinto del maestro): {distinto_del_maestro.sum():,} filas")
registrar("R-26", "D-18", "H-39, H-43", "Marcar", "Marca pozos con varios tipos de recurso o distinto del maestro; se usa el de producción",
          df["recurso_inconsistente"].sum())

### R-27 a R-29 · Decisiones sin acción sobre el dato (D-16, D-17, D-19)

Tres decisiones concluyeron que **el dato está bien**. Quedan en el registro con 0 filas afectadas:

- **D-16 (H-40):** los pozos con varias empresas son **cambios reales de operador** (venta de áreas en 2024–2025).
  Cada mes queda atribuido a la empresa que lo operó: es lo que ya dice el dato.
- **D-17 (H-44):** los pozos de Vaca Muerta marcados CONVENCIONAL son pozos verticales viejos: es un dato real.
  El EDA filtra `formprod` = VMUT **y** `tipo_de_recurso` = NO CONVENCIONAL.
- **D-19 (H-28):** la columna `id` (solo en 2024 y 2025) no se usa y no se lee.

In [ ]:
registrar("R-27", "D-16", "H-40", "Sin acción", "Cambio real de operador: cada mes queda con la empresa que lo operó", 0)
registrar("R-28", "D-17", "H-44", "Sin acción", "Vaca Muerta convencional es real; el EDA filtra VMUT y NO CONVENCIONAL", 0)
registrar("R-29", "D-19", "H-28", "Sin acción", "La columna id no se lee", 0)

## 5. Registro de reglas: el linaje completo

Esta tabla resume, en orden, todo lo que se le hizo al dato entre los raw y el archivo limpio.

In [ ]:
registro_reglas = pd.DataFrame(registro)
registro_reglas["pct"] = (registro_reglas["filas_afectadas"] / TOTAL_FILAS * 100).round(2)
registro_reglas

## 6. Validación

Antes de guardar, comprobamos que el resultado cumple lo prometido (paso 5 de la política: *validar*).
`assert` frena el notebook con un mensaje si algo no se cumple.

- **No se perdió ninguna fila** y la clave `idpozo` + `anio` + `mes` sigue siendo única.
- **La producción total es idéntica** a la de los raw: ni un m³ de diferencia.
- Las **marcas** son de tipo sí/no (`bool`) y no tienen vacíos.

In [ ]:
COLUMNAS_MARCA = ["tef_fuera_de_rango", "produccion_negativa", "fuera_del_maestro", "recurso_inconsistente"]

assert len(df) == TOTAL_FILAS, "Cambió la cantidad de filas"
assert not df.duplicated(["idpozo", "anio", "mes"]).any(), "La clave idpozo + anio + mes dejó de ser única"
assert df["prod_pet_m3"].sum() == total_raw["prod_pet"], "Cambió el total de petróleo"
assert df["prod_gas_miles_m3"].sum() == total_raw["prod_gas"], "Cambió el total de gas"
assert (df[COLUMNAS_MARCA].dtypes == bool).all(), "Alguna marca no es sí/no"

print(f"Filas: {len(df):,} (las mismas que los raw)  |  clave única: OK  |  totales de producción idénticos: OK")
print("\nFilas marcadas por columna:")
print(df[COLUMNAS_MARCA].sum().to_string())
print(f"\nFilas con al menos una marca: {df[COLUMNAS_MARCA].any(axis=1).sum():,} "
      f"({df[COLUMNAS_MARCA].any(axis=1).mean() * 100:.2f}%)")

## 7. Guardar el archivo limpio (Parquet)

Guardamos el resultado en `data/clean` con un nombre que indica su origen y su período (decisión D-21: **Parquet**).

**¿Qué es Parquet?** Un formato de archivo para tablas, como el CSV, pero **comprimido** y que **guarda el tipo de cada columna**
(las fechas siguen siendo fechas y las marcas siguen siendo sí/no al volver a leerlo). Se abre con pandas (`pd.read_parquet`)
o con herramientas como Power BI; **no se abre con doble clic en Excel**.

Dos ajustes para que el archivo pese menos (sin cambiar ningún dato):
- **Ordenamos las filas** por pozo y mes (`sort_values`). Así las filas de un mismo pozo quedan juntas y se comprimen mucho mejor,
  porque repiten la sigla, la empresa y el yacimiento. Además, el archivo queda más fácil de leer.
- Usamos la compresión **zstd**, que comprime más que la que trae Parquet por defecto.

En una prueba con datos inventados de 2,6 millones de filas, estos dos ajustes bajaron el tamaño de 93 MB a 46 MB.

Después:
- comprobamos que pese **menos de 100 MB** (regla 8 del proyecto: GitHub no acepta archivos más grandes);
- volvemos a calcular la huella de los **raw** para probar que no cambiaron;
- calculamos la huella del **limpio**, para registrarla en `data/clean/README.md`.

In [ ]:
df = df.sort_values(["idpozo", "fecha_mes"], ignore_index=True)
df.to_parquet(RUTA_CLEAN, index=False, compression="zstd")

tamanio_mb = os.path.getsize(RUTA_CLEAN) / 1e6
assert tamanio_mb < 100, f"El archivo pesa {tamanio_mb:.1f} MB: supera el límite de GitHub"

for nombre, huella in huellas_raw.items():
    assert huella_sha256(os.path.join(CARPETA_RAW, nombre)) == huella, f"¡{nombre} cambió!"

huella_clean = huella_sha256(RUTA_CLEAN)
print("Raw sin modificar: OK (misma huella que al inicio)")
print("Archivo limpio:", RUTA_CLEAN)
print(f"Tamaño: {tamanio_mb:.1f} MB")
print("Huella SHA-256 del limpio:", huella_clean)

## 8. Verificación: releer el archivo guardado

Leemos el Parquet recién guardado y comprobamos que es **idéntico** a la tabla en memoria:
mismas filas (en el mismo orden), mismas columnas, mismos tipos y mismos valores. `equals` compara las dos tablas celda por celda.

In [ ]:
limpio = pd.read_parquet(RUTA_CLEAN)

assert limpio.equals(df), "El archivo releído no es idéntico a la tabla en memoria"
print(f"Archivo releído: {limpio.shape[0]:,} filas x {limpio.shape[1]} columnas  |  idéntico a la tabla en memoria: OK")
print("\nColumnas y tipos:")
print(limpio.dtypes.to_string())

## 9. Conclusiones

- **No se eliminó ninguna fila ni se modificó ningún valor de producción:** los totales de petróleo y gas son idénticos a los de los raw.
- **Se corrigió el formato** (unidad en el nombre de las columnas, fecha mensual real) y **se marcaron** los casos dudosos en cuatro columnas sí/no.
- **El linaje está completo:** cada fila tiene su `archivo_origen`, y cada cambio está en el registro de reglas (R-21 a R-29) con la decisión y los hallazgos que lo justifican.
- **Integridad probada:** las huellas SHA-256 muestran que los raw no cambiaron; el archivo limpio queda identificado con su propia huella.

**Cómo usar el archivo limpio en el EDA**
- Petróleo y gas **no se suman**: tienen unidades distintas (`prod_pet_m3` y `prod_gas_miles_m3`).
- Para Vaca Muerta no convencional: `formprod == "VMUT"` y `tipo_de_recurso == "NO CONVENCIONAL"`.
- Las filas marcadas se **incluyen** en los totales (su efecto es nulo); si un análisis usa `tef`, debe excluir `tef_fuera_de_rango`.

**Etapa siguiente:** `05_eda.ipynb`, con los cuatro gráficos.